# Exploratory analysis
Inspect raw Uber Eats tables before modeling.

In [8]:
import pandas as pd
import numpy as np

In [5]:
from eats_intel.data import load_all

tables = load_all()
print(tables.keys())

for name, df in tables.items():
    print(f"{name}: {df.shape}")

dict_keys(['customers', 'restaurants', 'drivers', 'orders', 'deliveries', 'payments', 'reviews'])
customers: (8000, 16)
restaurants: (500, 14)
drivers: (400, 9)
orders: (50000, 20)
deliveries: (47851, 14)
payments: (50000, 10)
reviews: (19082, 9)


In [6]:
customers=tables['customers']
orders=tables['orders']
restaurants=tables['restaurants']
drivers=tables['drivers']
deliveries=tables['deliveries']
payments=tables['payments']
reviews=tables['reviews']

### Preparing Analytical Features for Customers

The following analytical features can be created to understand customer purchasing behavior, spending patterns, and ordering preferences.

- **Total Orders:** Total number of orders placed by each customer.
- **Average Order Value:** Average amount spent per order by each customer.
- **Total Spending:** Total amount spent by each customer across all orders.
- **Ordering Frequency:** How frequently each customer places orders within a given period.
- **Average Rating Given:** Average rating provided by each customer for their orders or experiences.
- **Weekend Orders:** Total number of orders placed by each customer on weekends.
- **Late-Night Orders:** Total number of orders placed by each customer during late-night hours.

In [68]:
#customer table eda 
#customers.info()
orders.info()
#restaurants.info()
#drivers.info()
#deliveries.info()
#reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           50000 non-null  str           
 1   customer_id        50000 non-null  str           
 2   restaurant_id      50000 non-null  str           
 3   order_timestamp    50000 non-null  datetime64[us]
 4   order_date         50000 non-null  object        
 5   order_hour         50000 non-null  int64         
 6   day_of_week        50000 non-null  str           
 7   is_weekend         50000 non-null  bool          
 8   city               50000 non-null  str           
 9   weather            50000 non-null  str           
 10  traffic_condition  50000 non-null  str           
 11  item_count         50000 non-null  int64         
 12  subtotal           50000 non-null  float64       
 13  delivery_fee       50000 non-null  float64       
 14  service_fee      

In [79]:
#total orders 
customer_orders = pd.merge(
    customers, orders,
    on="customer_id",
    how="left"
)

customers['total_orders'] = (
    customer_orders.groupby('customer_id')['order_id']
    .transform('count')
    .groupby(customer_orders['customer_id'])
    .first()
    .reindex(customers['customer_id'])
    .values
)
customers['total_orders']

0       2
1       9
2       6
3       4
4       4
       ..
7995    3
7996    7
7997    8
7998    9
7999    6
Name: total_orders, Length: 8000, dtype: int64

In [22]:
#Weekend orders 
weekend_orders = (
    orders[orders['is_weekend']]
    .groupby('customer_id')['order_id']
    .nunique()
    .rename('weekend_orders')
)
weekend_orders

customer_id
C000001    1
C000003    3
C000004    1
C000005    1
C000006    4
          ..
C007993    2
C007994    2
C007996    2
C007997    4
C007998    2
Name: weekend_orders, Length: 6689, dtype: int64

In [33]:
#Late night orders
orders['is_late_night'] = (
    (orders['order_hour'] >= 22) |
    (orders['order_hour'] < 5)
)

late_night_orders = (
    orders[orders['is_late_night']]
    .groupby('customer_id')['order_id']
    .nunique()
    .rename('late_night_orders')
)
late_night_orders

customer_id
C000005    1
C000006    1
C000012    1
C000020    1
C000022    1
          ..
C007982    2
C007986    2
C007991    1
C007997    1
C007998    1
Name: late_night_orders, Length: 2398, dtype: int64

In [32]:
#Average rating provided by customer
customer_orders = pd.merge(
    customers, reviews,
    on="customer_id",
    how="left"
)

customers['avg_rating'] = (
    customer_orders.groupby('customer_id')['rating']
    .transform('mean')
    .groupby(customer_orders['customer_id'])
    .first()
    .reindex(customers['customer_id'])
    .values
)
customers['avg_rating']

0        NaN
1       4.50
2       4.00
3       5.00
4       4.00
        ... 
7995    4.00
7996    4.25
7997    5.00
7998    4.25
7999    4.00
Name: avg_rating, Length: 8000, dtype: float64

In [37]:
#frequency per customer
period_months = (
    (orders['order_timestamp'].max().year - orders['order_timestamp'].min().year) * 12
    + orders['order_timestamp'].max().month
    - orders['order_timestamp'].min().month
    + 1
)

customers['ordering_frequency'] = (
    customers['total_orders'] / period_months
)
customers['ordering_frequency']

0       0.50
1       2.25
2       1.50
3       1.00
4       1.00
        ... 
7995    0.75
7996    1.75
7997    2.00
7998    2.25
7999    1.50
Name: ordering_frequency, Length: 8000, dtype: float64

In [48]:
# Calculate customer-level features
customer_features = orders.groupby('customer_id').agg(
    total_spending=('order_amount', 'sum'),
    average_order_value=('order_amount', 'mean')).reset_index()

# Remove previously calculated features, if they exist
features = [
    'total_spending',
    'average_order_value',
    'total_orders'
]

customers = customers.drop(
    columns=[col for col in features if col in customers.columns]
)

# Merge the updated features into customers
customers = customers.merge(
    customer_features,
    on='customer_id',
    how='left'
)

# Display column names
print(customers.columns.tolist())

# View the updated data
display(customers.head())

['customer_id', 'first_name', 'last_name', 'email', 'age', 'gender', 'city', 'neighborhood', 'latitude', 'longitude', 'signup_date', 'membership_tier', 'preferred_cuisine', 'device_type', 'account_status', 'lifetime_orders_prior', 'total_orders_x', 'avg_rating', 'ordering_frequency', 'total_spending_x', 'average_order_value_x', 'total_orders_y', 'total_spending_y', 'average_order_value_y', 'total_spending', 'average_order_value']


,customer_id,first_name,last_name,email,age,gender,city,neighborhood,latitude,longitude,...,total_orders_x,avg_rating,ordering_frequency,total_spending_x,average_order_value_x,total_orders_y,total_spending_y,average_order_value_y,total_spending,average_order_value
0,C000001,Danielle,Johnson,danielle.johnson0@example.com,44,Male,Austin,Domain,30.208602,-97.736069,...,2,NaN,0.50,114.19,57.095000,2.0,114.19,57.095000,114.19,57.095000
1,C000002,Jeffrey,Doyle,jeffrey.doyle1@example.com,46,Male,New York,Williamsburg,40.674132,-73.985719,...,9,4.5,2.25,496.46,55.162222,9.0,496.46,55.162222,496.46,55.162222
2,C000003,Patricia,Miller,patricia.miller2@example.com,32,Non-binary,Austin,Zilker,30.251354,-97.713823,...,6,4.0,1.50,423.04,70.506667,6.0,423.04,70.506667,423.04,70.506667
3,C000004,Amanda,Robinson,amanda.robinson3@example.com,25,Female,Austin,South Congress,30.318004,-97.749367,...,4,5.0,1.00,304.52,76.130000,4.0,304.52,76.130000,304.52,76.130000
4,C000005,Anthony,Gonzalez,anthony.gonzalez4@example.com,39,Male,New York,East Village,40.723247,-73.999582,...,4,4.0,1.00,234.81,58.702500,4.0,234.81,58.702500,234.81,58.702500


In [72]:
# Calculate restaurant-level features
average_delivery_time=deliveries.groupby("order_id")['delivery_time_min'].mean()
avg_prep_time=restaurants['avg_prep_time_min']
avg_rating_resto=reviews.groupby("restaurant_id")['rating'].mean()
cancellation_rate = (
    orders.assign(
        is_cancelled=orders['order_status'].str.lower().eq('cancelled')
    )
    .groupby('restaurant_id')['is_cancelled']
    .mean()
    .mul(100)
    .reset_index(name='cancellation_rate')
)
revenue_rest=orders.groupby('restaurant_id')['order_amount'].sum()
orders_rest=orders.groupby('restaurant_id')['order_id'].count()

In [73]:
# 1. Average delivery time per restaurant
average_delivery_time = (
    orders[['order_id', 'restaurant_id']]
    .merge(deliveries[['order_id', 'delivery_time_min']], on='order_id', how='left')
    .groupby('restaurant_id')['delivery_time_min']
    .mean()
    .rename('average_delivery_time')
)

# 2. Average preparation time per restaurant
avg_prep_time = (
    restaurants.set_index('restaurant_id')['avg_prep_time_min']
    .rename('average_preparation_time')
)

# 3. Average rating per restaurant
avg_rating_resto = (
    reviews.groupby('restaurant_id')['rating']
    .mean()
    .rename('average_rating')
)

# 4. Cancellation rate per restaurant (%)
cancellation_rate = (
    orders.assign(
        is_cancelled=orders['order_status'].str.lower().eq('cancelled')
    )
    .groupby('restaurant_id')['is_cancelled']
    .mean()
    .mul(100)
    .rename('cancellation_rate')
)

# 5. Revenue per restaurant
revenue_rest = (
    orders.groupby('restaurant_id')['order_amount']
    .sum()
    .rename('revenue')
)

# 6. Total orders per restaurant
orders_rest = (
    orders.groupby('restaurant_id')['order_id']
    .nunique()
    .rename('total_orders')
)

# Combine all features by restaurant_id
resto_features = pd.concat(
    [
        average_delivery_time,
        avg_prep_time,
        avg_rating_resto,
        cancellation_rate,
        revenue_rest,
        orders_rest
    ],
    axis=1
).reset_index()

# Display the result
display(resto_features.head())

,restaurant_id,average_delivery_time,average_preparation_time,average_rating,cancellation_rate,revenue,total_orders
0,R00001,27.411538,23,3.857143,0.0,3750.95,83
1,R00002,19.442857,7,4.250000,0.0,4146.20,99
2,R00003,32.776852,13,3.972973,0.0,5127.45,120
3,R00004,27.022951,20,3.806452,0.0,2952.88,66
4,R00005,30.876259,16,3.931034,0.0,6957.29,152


In [78]:
# Save engineered feature tables
customers.to_csv('/home/ninelaps/uber-ml/data/csv/customers.csv', index=False)
resto_features.to_csv('/home/ninelaps/uber-ml/data/csv/restaurant_features.csv', index=False)

print("Feature engineering completed.")
print("Customer features:", customers.shape)
print("Restaurant features:", resto_features.shape)

Feature engineering completed.
Customer features: (8000, 28)
Restaurant features: (500, 7)


In [77]:
# Merge weekend orders and late-night orders
customers = customers.merge(
    weekend_orders,
    on='customer_id',
    how='left'
).merge(
    late_night_orders,
    on='customer_id',
    how='left'
)

# Replace missing values with 0
customers[['weekend_orders', 'late_night_orders']] = (
    customers[['weekend_orders', 'late_night_orders']]
    .fillna(0)
    .astype(int)
)

# Display results
display(customers.head())

,customer_id,first_name,last_name,email,age,gender,city,neighborhood,latitude,longitude,...,ordering_frequency,total_spending_x,average_order_value_x,total_orders_y,total_spending_y,average_order_value_y,total_spending,average_order_value,weekend_orders,late_night_orders
0,C000001,Danielle,Johnson,danielle.johnson0@example.com,44,Male,Austin,Domain,30.208602,-97.736069,...,0.50,114.19,57.095000,2.0,114.19,57.095000,114.19,57.095000,1,0
1,C000002,Jeffrey,Doyle,jeffrey.doyle1@example.com,46,Male,New York,Williamsburg,40.674132,-73.985719,...,2.25,496.46,55.162222,9.0,496.46,55.162222,496.46,55.162222,0,0
2,C000003,Patricia,Miller,patricia.miller2@example.com,32,Non-binary,Austin,Zilker,30.251354,-97.713823,...,1.50,423.04,70.506667,6.0,423.04,70.506667,423.04,70.506667,3,0
3,C000004,Amanda,Robinson,amanda.robinson3@example.com,25,Female,Austin,South Congress,30.318004,-97.749367,...,1.00,304.52,76.130000,4.0,304.52,76.130000,304.52,76.130000,1,0
4,C000005,Anthony,Gonzalez,anthony.gonzalez4@example.com,39,Male,New York,East Village,40.723247,-73.999582,...,1.00,234.81,58.702500,4.0,234.81,58.702500,234.81,58.702500,1,1
